# Equicorrelated d=3 probe
All three pairwise correlations equal rho (X_i = sqrt(rho) g + sqrt(1-rho) z_i).
No closed form is claimed for this geometry (sectors couple; H_13 is not
confined to sector 0). Pre-registered, PROVABLE anchors only are acceptance-checked:
rho=0 -> 1 for zero-mean product targets (independence argument), rho=1 -> 0
(degeneration to the one-variable function g^3 resp. tanh(g)^3), and a
pairwise control h = tanh(x1+x2)+tanh(x2+x3)+tanh(x1+x3) that lies in S_2
by construction, so F_2 = 0 EXACTLY at every rho (checked on the RBF
estimator; the degree-4 polynomial cannot represent tanh sums, so its
residual on this control is basis error and is recorded, not tested).
additive_index is retained as a DESCRIPTIVE, unchecked target for
comparability with the single-pair experiment. Mid-range values are
RECORDED, not range-checked. The single-pair closed form
F(rho) is printed as a reference column and explicitly does NOT apply here.
Outputs to `MyDrive/KDD_Interactions/results/equicorr_probe/`.


In [ ]:
# Cell 1 -- Mount Drive and set up output folder
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/KDD_Interactions'
OUT = os.path.join(BASE, 'results', 'equicorr_probe')
os.makedirs(OUT, exist_ok=True)
print('output folder:', OUT)


In [ ]:
# Cell 2 -- Data generator (equicorrelated) + estimators (identical to order_probe_v2)
import numpy as np, json, csv, time, hashlib, os
from itertools import product as iproduct

def make_data_equicorr(n, rho, seed, target):
    """Equicorrelated Gaussian: corr(X_i, X_j) = rho for all pairs,
    via X_i = sqrt(rho) g + sqrt(1-rho) z_i. Valid for rho in [0, 1]."""
    # NOTE rho=1: all three columns coincide (design exactly singular);
    # intended -- ridge and lstsq both handle the degeneracy.
    rng = np.random.default_rng(seed)
    g = rng.standard_normal(n)
    z = rng.standard_normal((3, n))
    X = np.sqrt(rho) * g + np.sqrt(1 - rho) * z   # (3, n)
    X = X.T
    x1, x2, x3 = X[:, 0], X[:, 1], X[:, 2]
    if target == "monomial":
        h = x1 * x2 * x3
    elif target == "tanh_prod":
        h = np.tanh(x1) * np.tanh(x2) * np.tanh(x3)
    elif target == "pairwise_control":
        # lies in S_2 by construction: population F_2 = 0 exactly at every rho;
        # the acceptance threshold below is applied to the RBF estimate.
        h = np.tanh(x1 + x2) + np.tanh(x2 + x3) + np.tanh(x1 + x3)
    elif target == "additive_index":
        # DESCRIPTIVE ONLY (no acceptance check): sigmoid of the sum has a
        # small but genuine third-order ANOVA component, so it is not a
        # clean zero-3-way control; retained for comparability with the
        # single-pair experiment (order_probe_v2).
        h = 1.0 / (1.0 + np.exp(-(x1 + x2 + x3)))
    else:
        raise ValueError(f"Unknown target: {target}")
    return X, h

# ---------- estimators: byte-identical logic to order_probe_v2 ----------
def monomial_exps(n_vars, D, max_active=2):
    out = []
    for combo in iproduct(range(D + 1), repeat=n_vars):
        if sum(combo) <= D and sum(1 for c in combo if c > 0) <= max_active:
            out.append(combo)
    return out

def frac_poly(X, h, D, max_active=2):
    exps = monomial_exps(3, D, max_active)
    cols = []
    for e in exps:
        col = np.ones(X.shape[0])
        for j, p in enumerate(e):
            if p > 0:
                col = col * X[:, j] ** p
        cols.append(col)
    Phi = np.column_stack(cols)
    mu = Phi.mean(0); sd = Phi.std(0); sd[sd == 0] = 1.0
    Phi = (Phi - mu) / sd
    const_idx = exps.index((0, 0, 0))
    Phi[:, const_idx] = 1.0
    hc = h - h.mean()
    denom = float(hc @ hc)
    if denom <= 0:
        return float("nan")
    beta, *_ = np.linalg.lstsq(Phi, hc, rcond=None)
    r = hc - Phi @ beta
    return float((r @ r) / denom)

CENTERS = np.linspace(-2.5, 2.5, 9)
BW = 0.75

def uni_feats(x):
    return np.column_stack([x] + [np.exp(-0.5 * ((x - c) / BW) ** 2) for c in CENTERS])

def design_rbf(X):
    n = X.shape[0]
    U = [uni_feats(X[:, j]) for j in range(3)]
    cols = [np.ones((n, 1))] + U
    for a, b in [(0, 1), (0, 2), (1, 2)]:
        cols.append((U[a][:, :, None] * U[b][:, None, :]).reshape(n, -1))
    return np.concatenate(cols, axis=1)

def frac_rbf_ridge(X, h, lam=10.0, split_seed=0, train_frac=0.75):
    """Holdout test-residual variance fraction under the RBF pairwise basis.
    Ridge does not penalize the intercept."""
    n = X.shape[0]
    idx = np.random.default_rng(split_seed).permutation(n)
    tr, te = idx[: int(train_frac * n)], idx[int(train_frac * n):]
    Phi = design_rbf(X)
    mu = Phi[tr].mean(0); sd = Phi[tr].std(0); sd[sd == 0] = 1.0
    Phi = (Phi - mu) / sd
    Phi[:, 0] = 1.0
    hm = h[tr].mean()
    P = np.eye(Phi.shape[1]); P[0, 0] = 0.0
    A = Phi[tr].T @ Phi[tr] + lam * P
    b = Phi[tr].T @ (h[tr] - hm)
    try:
        from scipy.linalg import cho_factor, cho_solve
        beta = cho_solve(cho_factor(A), b)
    except Exception:
        beta = np.linalg.solve(A, b)
    resid = (h[te] - hm) - Phi[te] @ beta
    denom = np.sum((h[te] - h[te].mean()) ** 2)
    if denom <= 0:
        return float("nan")
    return float((resid @ resid) / denom)

N = 60_000
SEEDS = [0, 1, 2]
RHOS = [0.0, 0.3, 0.5, 0.7, 0.9, 0.99, 1.0]
TARGETS = ["monomial", "tanh_prod", "pairwise_control", "additive_index"]
POLY_DEGS = [3, 4]
RIDGE_LAM = 10.0

# Provenance hash: function bytecode PLUS all configuration constants
# (bytecode alone does not capture changes to N, grids, CENTERS, BW, etc.)
_config = {"N": N, "SEEDS": SEEDS, "RHOS": RHOS, "TARGETS": TARGETS,
           "POLY_DEGS": POLY_DEGS, "RIDGE_LAM": RIDGE_LAM,
           "CENTERS": CENTERS.tolist(), "BW": BW, "train_frac": 0.75}
CODE_SHA = hashlib.sha256(
    b"".join(f.__code__.co_code for f in
             [make_data_equicorr, monomial_exps, frac_poly, uni_feats,
              design_rbf, frac_rbf_ridge])
    + json.dumps(_config, sort_keys=True).encode()).hexdigest()
print("provenance sha256 (bytecode + config):", CODE_SHA)


In [ ]:
# Cell 3 -- Sweep, results to Drive
EXP = "equicorr_probe"
t0 = time.time()
rows = []
for target in TARGETS:
    for rho in RHOS:
        for s in SEEDS:
            X, h = make_data_equicorr(N, rho, s, target)
            rec = {"experiment": EXP, "target": target, "rho": rho, "seed": s}
            for D in POLY_DEGS:
                rec[f"frac_poly_D{D}"] = frac_poly(X, h, D)
            rec["frac_rbf_ridge"] = frac_rbf_ridge(X, h, lam=RIDGE_LAM, split_seed=s)
            rows.append(rec)
        print(f"{target:15s} rho={rho:4.2f} done  ({time.time()-t0:6.1f}s)", flush=True)

with open(os.path.join(OUT, "per_seed.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)

import collections
summ = collections.defaultdict(list)
for r in rows:
    summ[(r["target"], r["rho"])].append(r)
sum_rows = []
for (target, rho), rs in summ.items():
    rec = {"experiment": EXP, "target": target, "rho": rho, "n_seeds": len(rs)}
    for col in [f"frac_poly_D{D}" for D in POLY_DEGS] + ["frac_rbf_ridge"]:
        vals = [r[col] for r in rs]
        rec[col + "_mean"] = float(np.mean(vals))
        rec[col + "_sd"] = float(np.std(vals))
    sum_rows.append(rec)
with open(os.path.join(OUT, "results.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(sum_rows[0].keys()))
    w.writeheader(); w.writerows(sum_rows)

with open(os.path.join(OUT, "metadata.json"), "w") as f:
    json.dump({"experiment": EXP, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
               "geometry": "equicorrelated: corr(Xi,Xj)=rho for all pairs",
               "N": N, "seeds": SEEDS, "rhos": RHOS, "targets": TARGETS,
               "poly_degs": POLY_DEGS, "ridge_lam": RIDGE_LAM,
               "code_sha256": CODE_SHA, "numpy": np.__version__}, f, indent=2)
print("wrote per_seed.csv, results.csv, metadata.json")


In [ ]:
# Cell 4 -- Verification from disk: provable anchors only; mid-range recorded
import csv as _csv
rows = list(_csv.DictReader(open(os.path.join(OUT, "per_seed.csv"))))
assert all(r["experiment"] == "equicorr_probe" for r in rows), "experiment stamp mismatch (stale file?)"

def agg(t, rho, col):
    v = [float(r[col]) for r in rows if r["target"] == t and abs(float(r["rho"]) - rho) < 1e-9]
    return float(np.mean(v)), float(np.std(v))

single_pair_F = lambda r: (1 - r**2) ** 2 / ((1 + r**2) * (1 + 2 * r**2))

print(f"{'target':15s}{'rho':>5s}  {'poly D=4':>15s}  {'RBF ridge':>15s}  {'single-pair F (ref only)':>25s}")
for t in TARGETS:
    for rho in RHOS:
        m4, s4 = agg(t, rho, "frac_poly_D4")
        mr, sr = agg(t, rho, "frac_rbf_ridge")
        ref = f"{single_pair_F(rho):12.4f}" if t in ("monomial", "tanh_prod") else "         --"
        print(f"{t:15s}{rho:5.2f}  {m4:7.4f}+-{s4:6.4f}  {mr:7.4f}+-{sr:6.4f}  {ref:>25s}")
    print()

checks = []
m, _ = agg("monomial", 0.0, "frac_poly_D4");   checks.append(("monomial rho=0 -> ~1 (independence anchor, provable)", 0.97 < m < 1.005))
m, _ = agg("monomial", 1.0, "frac_poly_D4");   checks.append(("monomial rho=1 -> ~0 (degenerates to g^3, one variable)", m < 0.02))
m, _ = agg("tanh_prod", 0.0, "frac_rbf_ridge"); checks.append(("tanh_prod rho=0 -> ~1 (zero-mean product anchor)", 0.95 < m < 1.10))
m, _ = agg("tanh_prod", 1.0, "frac_rbf_ridge"); checks.append(("tanh_prod rho=1 -> ~0 (degenerates to tanh(g)^3)", m < 0.02))
pc_ok = all(agg("pairwise_control", r, "frac_rbf_ridge")[0] < 0.02 for r in RHOS)
checks.append(("pairwise_control in S_2 by construction: RBF fraction < 0.02 at all rho (provable zero)", pc_ok))

# observations (NOT acceptance tests): monotone decay; faster-than-single-pair
mono = [agg("monomial", r, "frac_poly_D4")[0] for r in RHOS]
obs1 = all(mono[i] >= mono[i+1] - 1e-3 for i in range(len(mono)-1))
obs2 = all(agg("monomial", r, "frac_poly_D4")[0] <= single_pair_F(r) + 0.02 for r in RHOS[1:-1])
story = []
for name, ok in checks:
    line = ("PASS  " if ok else "FAIL  ") + name
    story.append(line); print(line)
story.append(f"OBS   monotone decay across grid (monomial, poly D4): {obs1}")
story.append(f"OBS   at or below single-pair law at equal pairwise rho (mid-range): {obs2}")
pd4 = [agg("pairwise_control", r, "frac_poly_D4")[0] for r in RHOS]
story.append("OBS   pairwise_control poly-D4 residual (basis error on tanh sums, recorded not tested): "
             + " ".join(f"{v:.4f}" for v in pd4))
ai = [agg("additive_index", r, "frac_rbf_ridge")[0] for r in RHOS]
story.append("OBS   additive_index (descriptive, unchecked): "
             + " ".join(f"{v:.4f}" for v in ai))
print(story[-2]); print(story[-1])
with open(os.path.join(OUT, "check.txt"), "w") as f:
    f.write("\n".join(story) + "\n")
print("\nwrote check.txt")
